# Real vs Fake Image Classification — Milestone 1

## Project Goal
Classify images as real or AI-generated (fake) using a CNN.

In [1]:
# Install dependencies
!pip install -q torch torchvision numpy matplotlib tqdm scikit-learn

In [4]:
# Set data path: "." for local or Colab with data in repo. For Colab+Drive, mount and set path.
DATA_DIR = "."

Classes: ['fake', 'real']


In [5]:
import torch
from src.dataset import get_dataloaders
from src.model import get_model
from src.train import train_full, validate
from src.evaluate import evaluate, get_metrics, plot_confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

BATCH_SIZE = 32
IMAGE_SIZE = 128
train_loader, test_loader, classes = get_dataloaders(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, num_workers=0
)
print("Classes:", classes, "| Train batches:", len(train_loader), "| Test batches:", len(test_loader))

Train:   0%|          | 0/1500 [00:00<?, ?it/s]c:\Users\leyla\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Train:   1%|          | 11/1500 [01:07<2:32:25,  6.14s/it, acc=53.4, loss=0.625]


KeyboardInterrupt: 

In [ ]:
## Experiment 1: Baseline CNN

Train baseline CNN (5 epochs; increase for better results).

In [ ]:
model1 = get_model(num_classes=2, dropout=0.3)
history1 = train_full(
    model1, train_loader, test_loader,
    num_epochs=5, lr=1e-3, device=device,
    save_path="checkpoints/exp1_baseline.pth"
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history1["train_loss"], label="Train")
axes[0].plot(history1["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(history1["train_acc"], label="Train")
axes[1].plot(history1["val_acc"], label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].set_title("Accuracy")
plt.tight_layout()
plt.show()

In [ ]:
# Load best model and evaluate
model1.load_state_dict(torch.load("checkpoints/exp1_baseline.pth", map_location=device)["model_state_dict"])
y_true, y_pred, y_probs = evaluate(model1, test_loader, device)
acc, cm, report = get_metrics(y_true, y_pred, list(classes))
print("Test Accuracy:", acc)
print("Confusion Matrix:\n", cm)
print(report)
plot_confusion_matrix(cm, list(classes))